# GLP
Sergio Cabrales

https://www.sac2.com/

# Setup

In [ ]:
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from datetime import datetime
import re

Documento borrador Plan Indicativo de Abastecimiento de GLP 2025-2034

https://www.upme.gov.co/simec/hidrocarburos/glp/


Gasnova

https://www.gasnova.co/informes-estadisticos/

## Datos

In [ ]:
import subprocess

url = 'https://www.gasnova.co/wp-content/uploads/2026/08/6.Informe-Estadistico-Junio-2026.xlsx'
output_filename = 'Informe-Estadistico.xlsx'

try:
    subprocess.run(['wget', url, '-O', output_filename], check=True)
    print(f'File downloaded successfully to {output_filename}')
except subprocess.CalledProcessError as e:
    print(f'Error downloading the file: {e}')
except FileNotFoundError:
    print('wget command not found. Please ensure it is installed and in your PATH.')

In [ ]:
excel_path_new = 'Informe-Estadistico.xlsx'
xls_new = pd.ExcelFile(excel_path_new)
sheet_names_new = xls_new.sheet_names
df_oferta_nacional = xls_new.parse('15. Oferta nacional de GLP')
# Display rows 23 to 29 (index 22 to 28)
df_oferta_nacional = df_oferta_nacional.iloc[20:28]
df_oferta_nacional

In [ ]:
xls_new = pd.ExcelFile(excel_path_new)
sheet_names_new = xls_new.sheet_names
df_fuente = xls_new.parse('15. Oferta nacional de GLP')
# Display rows 23 to 29 (index 22 to 28)
df_fuente = df_fuente.iloc[8:18]
df_fuente

In [ ]:
df_raw = pd.read_excel(excel_path_new,
                       sheet_name='6. Top 10 productores de GLP', header=None)

years = [int(y) for y in df_raw.iloc[7, 1:20].tolist()]

countries = {
    'Estados Unidos':         df_raw.iloc[8,  1:20].tolist(),
    'China':                  df_raw.iloc[9,  1:20].tolist(),
    'Arabia Saudita':         df_raw.iloc[10, 1:20].tolist(),
    'Rusia':                  df_raw.iloc[11, 1:20].tolist(),
    'Canadá':                 df_raw.iloc[12, 1:20].tolist(),
    'India':                  df_raw.iloc[13, 1:20].tolist(),
    'Emiratos Árabes Unidos': df_raw.iloc[14, 1:20].tolist(),
    'Qatar':                  df_raw.iloc[15, 1:20].tolist(),
    'Argelia':                df_raw.iloc[16, 1:20].tolist(),
    'Irán':                   df_raw.iloc[17, 1:20].tolist(),
    'Resto del mundo':        df_raw.iloc[18, 1:20].tolist(),
}
totals = df_raw.iloc[19, 1:20].tolist()

df_oferta_imp = pd.DataFrame(countries, index=years)
df_oferta_imp

In [ ]:
df_raw = pd.read_excel(excel_path_new,
                       sheet_name='8. Top 10 consumidores de GLP', header=None)

years = [int(y) for y in df_raw.iloc[7, 1:20].tolist()]

countries = {
    'China':                  df_raw.iloc[8,  1:20].tolist(),
    'Estados Unidos':         df_raw.iloc[9,  1:20].tolist(),
    'India':                  df_raw.iloc[10, 1:20].tolist(),
    'Arabia Saudita':         df_raw.iloc[11, 1:20].tolist(),
    'Japón':                  df_raw.iloc[12, 1:20].tolist(),
    'Rusia':                  df_raw.iloc[13, 1:20].tolist(),
    'Corea del Sur':          df_raw.iloc[14, 1:20].tolist(),
    'México':                 df_raw.iloc[15, 1:20].tolist(),
    'Indonesia':              df_raw.iloc[16, 1:20].tolist(),
    'Canadá':                 df_raw.iloc[17, 1:20].tolist(),
    'Resto del mundo':        df_raw.iloc[18, 1:20].tolist(),
}
totals = df_raw.iloc[19, 1:20].tolist()

df_demanda_int = pd.DataFrame(countries, index=years).fillna(0)
df_demanda_int

In [ ]:
# Usar la primera fila de df_oferta_nacional (que contiene 'Fuente [kg/mes]' y fechas) como encabezados
df_clean = df_oferta_nacional.copy()

# Obtener los nuevos nombres de columna de la primera fila y modificar el primero
new_columns = df_clean.iloc[0].tolist()
new_columns[0] = 'Offer_Type'  # Renombrar 'Fuente [kg/mes]' a 'Offer_Type'
df_clean.columns = new_columns

# Eliminar la fila que se usó como encabezado
df_clean = df_clean.iloc[1:].reset_index(drop=True)

# Establecer la columna 'Offer_Type' como índice temporal antes de transponer
df_clean = df_clean.set_index('Offer_Type')

# Transponer el DataFrame
df_oferta_T = df_clean.T.reset_index()

# Renombrar la primera columna del DataFrame transpuesto (que ahora contiene las fechas) a 'Date'
df_oferta_T.rename(columns={df_oferta_T.columns[0]: 'Date'}, inplace=True)

# Crear el DataFrame final, asegurando que la columna 'Date' sea datetime y el índice
df_oferta = df_oferta_T.copy()
df_oferta['Date'] = pd.to_datetime(df_oferta['Date'], errors='coerce')
df_oferta = df_oferta[df_oferta['Date'].notna()]  # Eliminar filas con fechas no válidas
df_oferta = df_oferta.set_index('Date')
df_oferta = df_oferta.sort_index()
df_oferta = df_oferta.fillna(0)
df_oferta = df_oferta.astype(float)

df_oferta

In [ ]:
excel_path_new = 'Informe-Estadistico.xlsx'
xls_new = pd.ExcelFile(excel_path_new)
sheet_names_new = xls_new.sheet_names
df_fuente = xls_new.parse('15. Oferta nacional de GLP')
df_fuente = df_fuente.iloc[8:18]

df_processed = df_fuente.copy()

# Get the first row to be used as header, convert to string, and strip whitespace
new_header = df_processed.iloc[0].astype(str).str.strip()

# Assign new header to columns
df_processed.columns = new_header

# Remove the first row as it's now the header
df_processed = df_processed[1:].copy()

# Ensure all column names are string type (especially the date columns)
df_processed.columns = df_processed.columns.astype(str)

# Rename the first column to 'Offer_Type'
# We are certain the first column is the one that needs to be 'Offer_Type' based on df_fuente structure
df_processed = df_processed.rename(columns={df_processed.columns[0]: 'Offer_Type'})

# Ensure the 'Offer_Type' column is truly 1-dimensional and contains string values
df_processed['Offer_Type'] = df_processed['Offer_Type'].astype(str)

# Set 'Offer_Type' as index
df_processed = df_processed.set_index('Offer_Type')

# Transpose the DataFrame
df_fuente_T = df_processed.T

# Rename the index to 'Date'
df_fuente_T.index.name = 'Date'

# Convert index to datetime
df_fuente_T.index = pd.to_datetime(df_fuente_T.index, errors='coerce')

# Drop rows where Date is invalid
df_fuente_T = df_fuente_T[df_fuente_T.index.notna()]

# Sort by Date
df_fuente_T = df_fuente_T.sort_index()

# Assign to df_fuente
df_fuente = df_fuente_T.copy()

# Ensure the column names are just simple strings and not a named Index
df_fuente.columns.name = None

# Fill NaN values with 0 and convert to float
df_fuente = df_fuente.fillna(0).astype(float)

df_fuente.tail(3)


## Capa canónica de datos y verificación de consistencia

In [ ]:
# ── Tema y paleta EIA (definidos una sola vez) ───────────────────────────────
import matplotlib as mpl
import matplotlib.dates as mdates


def apply_eia_theme():
    mpl.rcParams.update({
        "figure.facecolor":  "#FFFFFF",
        "axes.facecolor":    "#FFFFFF",
        "axes.edgecolor":    "#BDBDBD",
        "axes.labelcolor":   "#1A1A1A",
        "axes.grid":         True,
        "grid.color":        "#E0E0E0",
        "grid.linewidth":    0.7,
        "axes.grid.axis":    "y",
        "text.color":        "#1A1A1A",
        "xtick.color":       "#1A1A1A",
        "ytick.color":       "#1A1A1A",
        "xtick.labelsize":   8,
        "ytick.labelsize":   8,
        "font.family":       "sans-serif",
        "legend.facecolor":  "#FFFFFF",
        "legend.edgecolor":  "#BDBDBD",
        "legend.fontsize":   8,
        "figure.dpi":        150,
    })


EIA = {"blue1": "#1F5FA6", "blue2": "#4A90D9", "orange": "#E8821A",
       "green": "#2E7D32", "red": "#C62828", "gray": "#757575",
       "purple": "#9C27B0", "watermark": "#9E9E9E"}

apply_eia_theme()

# ── Capa canónica de datos: un solo origen para todas las gráficas de oferta ──
# Motivo: cada gráfica reconstruía la oferta con un subconjunto distinto de
# fuentes, por lo que los "totales" no coincidían entre láminas.

RUTA_GASNOVA = 'Informe-Estadistico.xlsx'
HOJA_OFERTA  = '15. Oferta nacional de GLP'

_raw15 = pd.read_excel(RUTA_GASNOVA, sheet_name=HOJA_OFERTA, header=None)

# Bloque A — producción Ecopetrol por planta (fechas fila 9; plantas filas 10-18)
FECHAS = pd.to_datetime(_raw15.iloc[9, 1:], errors='coerce')

plantas = _raw15.iloc[10:19, 1:].T
plantas.columns = [str(c).strip() for c in _raw15.iloc[10:19, 0]]
plantas.index = FECHAS
plantas = plantas.apply(pd.to_numeric, errors='coerce').fillna(0.0) / 1e3   # ton/mes

# Bloque B — oferta nacional total por fuente (fechas fila 9; fuentes filas 22-28)
fuentes = _raw15.iloc[22:29, 1:].T
fuentes.columns = [str(c).replace('  [Kg/Mes]', '').replace('Oferta ', '').strip()
                   for c in _raw15.iloc[22:29, 0]]
fuentes.index = FECHAS
fuentes = fuentes.apply(pd.to_numeric, errors='coerce').fillna(0.0) / 1e3   # ton/mes

# Calendario común: meses con dato en el bloque B
_meses_validos = fuentes.index.notna() & (fuentes.sum(axis=1) > 0)
plantas, fuentes = plantas[_meses_validos], fuentes[_meses_validos]

# Diccionario único de agrupaciones
CAMPOS      = ['Apiay', 'Cusiana', 'Cupiagua', 'Dina', 'Propilco', 'Floreña']
REFINERIAS  = ['Barranca', 'Cartagena']
IMPORTADO   = ['Ecopetrol Importado', 'Importaciones G5',
               'Importaciones Tumaco', 'Importaciones Plexa']

oferta = pd.DataFrame(index=fuentes.index)
oferta['Campos']            = plantas[CAMPOS].sum(axis=1)
oferta['Barranca']          = plantas['Barranca']
oferta['Cartagena']         = plantas['Cartagena']
oferta['Termoyopal']        = fuentes['Termoyopal']
oferta['Otros Productores'] = fuentes['Otros Productores']
oferta['Importado']         = fuentes[IMPORTADO].sum(axis=1)

# Desagregación nacional usada por TODAS las gráficas de oferta
NACIONAL_DESAGREGADO = ['Campos', 'Barranca', 'Cartagena',
                        'Termoyopal', 'Otros Productores']
oferta['Nacional']      = oferta[NACIONAL_DESAGREGADO].sum(axis=1)
oferta['Total']         = oferta['Nacional'] + oferta['Importado']
oferta['pct_importado'] = oferta['Importado'] / oferta['Total'] * 100

# Detalle de importaciones (mismo total que oferta['Importado'])
importaciones = fuentes[IMPORTADO].copy()

# ── Manejo explícito del año parcial ──────────────────────────────────────────
MESES_ES = ['enero', 'febrero', 'marzo', 'abril', 'mayo', 'junio',
            'julio', 'agosto', 'septiembre', 'octubre', 'noviembre', 'diciembre']

ULTIMO_MES   = oferta.index.max()
ANIO_PARCIAL = ULTIMO_MES.year
N_MESES_PARCIAL = int((oferta.index.year == ANIO_PARCIAL).sum())
ANIO_COMPLETO   = N_MESES_PARCIAL == 12


def anio_parcial(idx):
    """(año, nº de meses, ¿completo?) del último año de una serie mensual."""
    idx = pd.DatetimeIndex(idx)
    ult = idx.max()
    n = int((idx.year == ult.year).sum())
    return ult.year, n, ult, n == 12


def nota_parcial(idx):
    """Nota al pie derivada de los datos, no escrita a mano."""
    anio, n, ult, completo = anio_parcial(idx)
    if completo:
        return ''
    return f'  *{anio}: enero-{MESES_ES[ult.month - 1]} ({n} meses)'


def etiquetas_anuales(idx, referencia=None):
    """Etiqueta de año, marcando con * el año parcial de la serie mensual."""
    anio, _, _, completo = anio_parcial(referencia if referencia is not None
                                        else oferta.index)
    return [f.strftime('%Y') + ('*' if (f.year == anio and not completo) else '')
            for f in idx]

NOTA_PARCIAL = ''  # se fija abajo, tras definir nota_parcial()


def promedio_mensual_anual(df):
    """Promedio ton/mes por año: comparable aunque el último año esté incompleto."""
    return df.resample('YS').mean()

NOTA_PARCIAL = nota_parcial(oferta.index)

print(f'Serie de oferta: {oferta.index.min():%Y-%m} a {ULTIMO_MES:%Y-%m} '
      f'({len(oferta)} meses). Año parcial: {ANIO_PARCIAL} con {N_MESES_PARCIAL} meses.')
oferta.tail(3).round(0)

In [ ]:
# ── Verificación de consistencia (ejecutar tras cargar los datos) ────────────
# Comprueba que los dos bloques de la hoja 15 cuadran entre sí y que el total
# de cada gráfica se puede derivar del mismo agregado.

_chk = pd.DataFrame({
    'Ecopetrol Nacional (bloque B)': fuentes['Ecopetrol Nacional'],
    'Suma plantas (bloque A)':       plantas[CAMPOS + REFINERIAS].sum(axis=1),
    'Ecopetrol Importado (B)':       fuentes['Ecopetrol Importado'],
    'Terminal Fluvial (A)':          plantas['Terminal Fluvial'],
})
_chk['dif nacional'] = _chk.iloc[:, 0] - _chk.iloc[:, 1]
_chk['dif importado'] = _chk.iloc[:, 2] - _chk.iloc[:, 3]

_tol = 1.0  # toneladas/mes
_max_nac = _chk['dif nacional'].abs().max()
_max_imp = _chk['dif importado'].abs().max()
print(f'Máx. |Ecopetrol Nacional − Σ plantas|  = {_max_nac:,.2f} ton/mes')
print(f'Máx. |Ecopetrol Importado − Terminal Fluvial| = {_max_imp:,.2f} ton/mes')

if _max_nac > _tol:
    print('\nAVISO: los bloques no cuadran en estos meses '
          '(revisar filas de la hoja 15 antes de graficar):')
    print(_chk.loc[_chk['dif nacional'].abs() > _tol].round(1).to_string())

# El total de la oferta debe ser idéntico se sume por fuente o por desagregado
assert np.allclose(oferta['Total'], fuentes.sum(axis=1), atol=_tol), \
    'El total desagregado no coincide con el total por fuente'
assert np.allclose(oferta['Importado'], importaciones.sum(axis=1), atol=_tol)
print('\nOK: todas las gráficas de oferta parten del mismo total.')

# Tabla de reconciliación: promedio ton/mes por año
_recon = promedio_mensual_anual(
    oferta[NACIONAL_DESAGREGADO + ['Importado', 'Nacional', 'Total']])
_recon.index = etiquetas_anuales(_recon.index)
print('\nPromedio ton/mes por año (base común de todas las láminas):')
print(_recon.round(0).to_string())

# Mundial de GLP

## Producción mundial

In [ ]:
df_raw = pd.read_excel(excel_path_new,
                       sheet_name='6. Top 10 productores de GLP', header=None)

years = [int(y) for y in df_raw.iloc[7, 1:20].tolist()]

countries = {
    'Estados Unidos':         df_raw.iloc[8,  1:20].tolist(),
    'China':                  df_raw.iloc[9,  1:20].tolist(),
    'Arabia Saudita':         df_raw.iloc[10, 1:20].tolist(),
    'Rusia':                  df_raw.iloc[11, 1:20].tolist(),
    'Canadá':                 df_raw.iloc[12, 1:20].tolist(),
    'India':                  df_raw.iloc[13, 1:20].tolist(),
    'Emiratos Árabes Unidos': df_raw.iloc[14, 1:20].tolist(),
    'Qatar':                  df_raw.iloc[15, 1:20].tolist(),
    'Argelia':                df_raw.iloc[16, 1:20].tolist(),
    'Irán':                   df_raw.iloc[17, 1:20].tolist(),
    'Resto del mundo':        df_raw.iloc[18, 1:20].tolist(),
}

df_oferta_imp = pd.DataFrame(countries, index=years).apply(pd.to_numeric,
                                                          errors='coerce').fillna(0.0)

# El total rotulado debe ser el de las barras dibujadas, no la fila "Total" de
# la hoja: si no, los porcentajes no suman 100 % y la etiqueta no cuadra.
totals_arr  = df_oferta_imp.sum(axis=1).values
totals_hoja = pd.to_numeric(pd.Series(df_raw.iloc[19, 1:20].tolist()),
                            errors='coerce').fillna(0.0).values
_dif = np.abs(totals_arr - totals_hoja)
if _dif.max() > 1:
    print('AVISO: la fila "Total" de la hoja difiere de la suma de los países '
          f'graficados (máx. {_dif.max():,.0f} kt). Se usa la suma graficada.')

colors = [
    '#1f4e79', '#2e75b6', '#5ba3d9', '#aad4f0',
    '#c55a11', '#ed7d31', '#ffc000', '#70ad47',
    '#7030a0', '#00b0f0', 'darkgray'  # gris para Resto del mundo
]

fig, ax = plt.subplots(figsize=(13, 6))
fig.patch.set_facecolor('#f8f9fa')
ax.set_facecolor('#f8f9fa')

x = np.arange(len(years))
bar_width = 0.65
bottoms = np.zeros(len(years))

for country, color in zip(countries.keys(), colors):
    vals = df_oferta_imp[country].values
    bars = ax.bar(x, vals, bar_width, bottom=bottoms, color=color, label=country,
                  edgecolor='white', linewidth=0.4)

    for j, (bar, v) in enumerate(zip(bars, vals)):
        pct = v / totals_arr[j] * 100
        if pct >= 2:
            cy = bottoms[j] + v / 2
            text_color = 'white' if color not in ['#ffc000', '#aad4f0'] else '#1a1a1a'
            ax.text(bar.get_x() + bar.get_width() / 2, cy,
                    f'{pct:.1f}%',
                    ha='center', va='center', fontsize=6,
                    color=text_color)

    bottoms += vals

# Totales encima de cada barra (= altura real de la pila)
for j, tot in enumerate(totals_arr):
    ax.text(j, tot + totals_arr.max() * 0.01, f'{tot:,.0f}',
            ha='center', va='bottom', fontsize=6,
            color='#1a1a1a')

ax.set_xticks(x)
ax.set_xticklabels([str(y) for y in years], fontsize=8)
ax.set_ylabel('Miles de toneladas (kt)', fontsize=11, labelpad=10)
ax.set_title(f'Principales productores de GLP {years[0]}–{years[-1]} '
             '(miles de toneladas) @SergioCabrales',
             fontsize=13, pad=18, color='#1a1a1a')

ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda val, _: f'{int(val):,}'))
ax.set_ylim(0, totals_arr.max() * 1.10)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_color('#cccccc')
ax.spines['bottom'].set_color('#cccccc')
ax.tick_params(axis='both', colors='#555555')
ax.yaxis.grid(True, linestyle='--', alpha=0.5, color='#cccccc')
ax.set_axisbelow(True)

ax.legend(loc='upper left', fontsize=8.5, framealpha=0.9,
          edgecolor='#cccccc', ncol=2, columnspacing=1,
          handlelength=1.2, handletextpad=0.5)

ax.text(0.30, -0.1, 'Fuente: Statistical Review of Global LPG 2025, WLPGA y Argus Media | Actualizado: enero 2026',
        transform=ax.transAxes, fontsize=6)

plt.tight_layout()
plt.show()

## Demanda mundial de GLP

In [ ]:
df_raw = pd.read_excel(excel_path_new,
                       sheet_name='8. Top 10 consumidores de GLP', header=None)

years = [int(y) for y in df_raw.iloc[7, 1:20].tolist()]

countries = {
    'China':                  df_raw.iloc[8,  1:20].tolist(),
    'Estados Unidos':         df_raw.iloc[9,  1:20].tolist(),
    'India':                  df_raw.iloc[10, 1:20].tolist(),
    'Arabia Saudita':         df_raw.iloc[11, 1:20].tolist(),
    'Japón':                  df_raw.iloc[12, 1:20].tolist(),
    'Rusia':                  df_raw.iloc[13, 1:20].tolist(),
    'Corea del Sur':          df_raw.iloc[14, 1:20].tolist(),
    'México':                 df_raw.iloc[15, 1:20].tolist(),
    'Indonesia':              df_raw.iloc[16, 1:20].tolist(),
    'Canadá':                 df_raw.iloc[17, 1:20].tolist(),
    'Resto del mundo':        df_raw.iloc[18, 1:20].tolist(),
}

df_demanda_int = pd.DataFrame(countries, index=years).apply(pd.to_numeric,
                                                            errors='coerce').fillna(0.0)

# Igual que en producción: el total rotulado es la suma de lo graficado.
totals_arr  = df_demanda_int.sum(axis=1).values
totals_hoja = pd.to_numeric(pd.Series(df_raw.iloc[19, 1:20].tolist()),
                            errors='coerce').fillna(0.0).values
_dif = np.abs(totals_arr - totals_hoja)
if _dif.max() > 1:
    print('AVISO: la fila "Total" de la hoja difiere de la suma de los países '
          f'graficados (máx. {_dif.max():,.0f} kt). Se usa la suma graficada.')

colors = [
    '#1f4e79', '#2e75b6', '#5ba3d9', '#aad4f0',
    '#c55a11', '#ed7d31', '#ffc000', '#70ad47',
    '#7030a0', '#00b0f0', 'darkgray'
]

fig, ax = plt.subplots(figsize=(13, 6))
fig.patch.set_facecolor('#f8f9fa')
ax.set_facecolor('#f8f9fa')

x = np.arange(len(years))
bar_width = 0.65
bottoms = np.zeros(len(years))

for country, color in zip(countries.keys(), colors):
    vals = df_demanda_int[country].values
    bars = ax.bar(x, vals, bar_width, bottom=bottoms, color=color, label=country,
                  edgecolor='white', linewidth=0.4)

    for j, (bar, v) in enumerate(zip(bars, vals)):
        if totals_arr[j] == 0:
            continue
        pct = v / totals_arr[j] * 100
        if pct >= 2:
            cy = bottoms[j] + v / 2
            text_color = 'white' if color not in ['#ffc000', '#aad4f0'] else '#1a1a1a'
            ax.text(bar.get_x() + bar.get_width() / 2, cy,
                    f'{pct:.1f}%',
                    ha='center', va='center', fontsize=6,
                    color=text_color)

    bottoms += vals

for j, tot in enumerate(totals_arr):
    ax.text(j, tot + totals_arr.max() * 0.01, f'{tot:,.0f}',
            ha='center', va='bottom', fontsize=6,
            color='#1a1a1a')

ax.set_xticks(x)
ax.set_xticklabels([str(y) for y in years], fontsize=9)
ax.set_ylabel('Miles de toneladas (kt)', fontsize=11, labelpad=10)
ax.set_title(f'Principales consumidores internacionales de GLP {years[0]}–{years[-1]} '
             '(miles de toneladas) @SergioCabrales',
             fontsize=13, pad=18, color='#1a1a1a')

ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda val, _: f'{int(val):,}'))
ax.set_ylim(0, totals_arr.max() * 1.10)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_color('#cccccc')
ax.spines['bottom'].set_color('#cccccc')
ax.tick_params(axis='both', colors='#555555')
ax.yaxis.grid(True, linestyle='--', alpha=0.5, color='#cccccc')
ax.set_axisbelow(True)

ax.legend(loc='upper left', fontsize=8.5, framealpha=0.9,
          edgecolor='#cccccc', ncol=2, columnspacing=1,
          handlelength=1.2, handletextpad=0.5)

ax.text(0.30, -0.1, 'Fuente: Statistical Review of Global LPG 2025, WLPGA y Argus Media | Actualizado: enero 2026',
        transform=ax.transAxes, fontsize=6)

plt.tight_layout()
plt.show()

# Colombia

## Producción mensual de GLP

In [ ]:
# Oferta mensual: nacional vs. importado (todo desde `oferta`, en ton/mes)
fig, ax = plt.subplots(figsize=(12, 7))

x_labels = oferta.index.strftime('%Y-%m')

ax.bar(x_labels, oferta['Nacional'], label='Producción Nacional',
       color='#005F73', alpha=0.6, width=0.8)
ax.bar(x_labels, oferta['Importado'], bottom=oferta['Nacional'],
       label='Importado', color='orange', alpha=0.6, width=0.8)

# % importado del último mes
_i = -1
if oferta['Importado'].iloc[_i] > 0:
    ax.text(len(oferta) - 1,
            oferta['Total'].iloc[_i] + oferta['Total'].max() * 0.01,
            f"{oferta['pct_importado'].iloc[_i]:.0f}%",
            ha='center', va='bottom', fontsize=9, color='black')

ax.set_title('Oferta Mensual de GLP en Colombia @SergioCabrales', fontsize=13, pad=14)
ax.set_ylabel('Toneladas / Mes', fontsize=11)

ax.spines['right'].set_visible(False)
ax.spines['top'].set_visible(False)
ax.grid(axis='y', linestyle='--', alpha=0.4, color='grey')
ax.set_axisbelow(True)

plt.xticks(rotation=90, ha='right', fontsize=8)
plt.yticks(fontsize=10)
ax.legend(frameon=False, loc='upper left', fontsize=10)
ax.set_ylim(0, oferta['Total'].max() * 1.10)

plt.tight_layout()
fig.text(0.01, 0.005, 'Fuente: Gasnova, hoja "15. Oferta nacional de GLP"',
         fontsize=7, color='grey')
plt.show()

print(f"Total último mes ({ULTIMO_MES:%Y-%m}): {oferta['Total'].iloc[-1]:,.0f} ton/mes")

In [ ]:
# Misma oferta que la lámina anterior, abierta por fuente.
# El total de esta gráfica coincide, mes a mes, con el de la anterior.
fig, ax = plt.subplots(figsize=(13, 7))

x_labels = oferta.index.strftime('%Y-%m')

COMPONENTES = NACIONAL_DESAGREGADO + ['Importado']
COLORES = {
    'Campos':            '#1F77B4',
    'Barranca':          'darkred',
    'Cartagena':         'darkgreen',
    'Termoyopal':        '#8C564B',
    'Otros Productores': '#7F7F7F',
    'Importado':         '#FF7F0E',
}
ETIQUETAS = {'Barranca': 'Refinería Barranca', 'Cartagena': 'Refinería Cartagena'}

bottom = np.zeros(len(oferta))
for comp in COMPONENTES:
    vals = oferta[comp].values
    ax.bar(x_labels, vals, bottom=bottom, width=0.8, alpha=0.6,
           color=COLORES[comp], label=ETIQUETAS.get(comp, comp))
    bottom += vals

# % importado del último mes
if oferta['Importado'].iloc[-1] > 0:
    ax.text(len(oferta) - 1,
            oferta['Total'].iloc[-1] + oferta['Total'].max() * 0.02,
            f"{oferta['pct_importado'].iloc[-1]:.0f}%",
            ha='center', va='bottom', fontsize=9, color='black')

ax.set_title('Oferta Mensual de GLP en Colombia — Comparación por Fuente @SergioCabrales',
             fontsize=13, pad=14)
ax.set_ylabel('Toneladas / Mes', fontsize=11)

ax.spines['right'].set_visible(False)
ax.spines['top'].set_visible(False)
ax.spines['left'].set_alpha(0.3)
ax.spines['bottom'].set_alpha(0.3)
ax.grid(axis='y', linestyle='--', linewidth=0.6, alpha=0.4, color='grey')
ax.set_axisbelow(True)

plt.xticks(rotation=90, ha='center', fontsize=7.5)
plt.yticks(fontsize=10)
ax.legend(frameon=False, loc='upper left', fontsize=10, ncol=2,
          title='Fuente de la oferta', title_fontsize=10)
ax.set_ylim(0, oferta['Total'].max() * 1.15)

plt.tight_layout()
fig.text(0.01, 0.005,
         'Fuente: Gasnova. Campos = Apiay, Cusiana, Cupiagua, Dina, Propilco y Floreña',
         fontsize=7, color='grey')
plt.show()

# Control: la pila reproduce exactamente el total de la lámina anterior
assert np.allclose(bottom, oferta['Total'].values, atol=1e-6)
print('OK: el total apilado coincide con oferta["Total"].')

In [ ]:
# Oferta mensual de Ecopetrol (campos + refinerías).
# OJO: es un subconjunto de la oferta nacional — no incluye Termoyopal,
# Otros Productores ni importaciones. Su total NO debe compararse con el de
# las láminas de oferta total.
apply_eia_theme()

ecp = pd.DataFrame({
    'Campos':    oferta['Campos'],
    'Barranca':  oferta['Barranca'],
    'Cartagena': oferta['Cartagena'],
})
df_plot = ecp[ecp.index >= '2020-01-01']
grupos  = ['Campos', 'Barranca', 'Cartagena']
colores = [EIA['blue1'], EIA['orange'], EIA['gray']]

fig, ax = plt.subplots(figsize=(11, 6))

bottom = np.zeros(len(df_plot))
for grupo, color in zip(grupos, colores):
    vals = df_plot[grupo].values
    ax.bar(df_plot.index, vals, bottom=bottom, color=color, label=grupo, width=22)
    bottom += vals

ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax.xaxis.set_minor_locator(mdates.MonthLocator())
ax.set_ylabel('Toneladas / mes', fontsize=9)
totales = df_plot.sum(axis=1)
ax.set_ylim(0, totales.max() * 1.15)
ax.legend(ncol=3, loc='upper right', frameon=False)

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_color('#BDBDBD')
ax.spines['bottom'].set_color('#BDBDBD')
ax.yaxis.grid(True, linestyle='--', alpha=0.5, color='gray')

ax.set_title('Oferta mensual de GLP de Ecopetrol: campos y refinerías @SergioCabrales',
             fontsize=13, color='#1A1A1A', pad=10)

fig.text(0.01, 0.01,
         'Fuente: Ecopetrol, cálculos GASNOVA. Campos: Apiay, Cusiana, Cupiagua, Dina, '
         'Propilco y Floreña. Excluye Termoyopal, otros productores e importaciones.',
         fontsize=7, color=EIA['gray'])

plt.tight_layout()
plt.show()

print(f'Ecopetrol {ULTIMO_MES:%Y-%m}: {totales.iloc[-1]:,.0f} ton/mes '
      f'({totales.iloc[-1] / oferta["Total"].iloc[-1] * 100:.1f} % de la oferta total)')

In [ ]:
# Ecopetrol por año. Se grafica el PROMEDIO ton/mes para que el año parcial
# sea comparable con los años completos (la suma anual lo hundiría).
apply_eia_theme()

ecp = pd.DataFrame({
    'Campos':    oferta['Campos'],
    'Barranca':  oferta['Barranca'],
    'Cartagena': oferta['Cartagena'],
})
df_anual = promedio_mensual_anual(ecp)
grupos   = ['Campos', 'Barranca', 'Cartagena']
colores  = [EIA['blue1'], EIA['orange'], EIA['gray']]

fig, ax = plt.subplots(figsize=(11, 6))

x = np.arange(len(df_anual))
bottom = np.zeros(len(df_anual))
for grupo, color in zip(grupos, colores):
    vals = df_anual[grupo].values
    ax.bar(x, vals, bottom=bottom, color=color, label=grupo, width=0.7)
    for xi, v, b in zip(x, vals, bottom):
        if v > df_anual.sum(axis=1).max() * 0.04:
            ax.text(xi, b + v / 2, f'{v:,.0f}', ha='center', va='center',
                    fontsize=7, color='white', fontweight='bold')
    bottom += vals

totales = df_anual[grupos].sum(axis=1).values
for xi, t in zip(x, totales):
    ax.text(xi, t + totales.max() * 0.015, f'{t:,.0f}',
            ha='center', va='bottom', fontsize=7.5, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(etiquetas_anuales(df_anual.index))
ax.set_ylabel('Toneladas / mes (promedio del año)', fontsize=9)
ax.set_ylim(0, totales.max() * 1.14)
ax.legend(ncol=3, loc='upper right', frameon=False)

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_color('#BDBDBD')
ax.spines['bottom'].set_color('#BDBDBD')
ax.yaxis.grid(True, linestyle='--', alpha=0.5, color='gray')

ax.set_title('Oferta de GLP de Ecopetrol: campos y refinerías @SergioCabrales',
             fontsize=13, color='#1A1A1A', pad=10)

fig.text(0.01, 0.01,
         'Fuente: Ecopetrol, cálculos GASNOVA. Campos: Apiay, Cusiana, Cupiagua, Dina, '
         'Propilco y Floreña.' + NOTA_PARCIAL,
         fontsize=7, color=EIA['gray'])

plt.tight_layout()
plt.show()

In [ ]:
# Oferta nacional mensual por fuente (bloque B de la hoja 15).
# Mismo total que las láminas de oferta nacional vs. importado.
apply_eia_theme()

df_plot = fuentes[fuentes.index >= '2020-01-01']
lista_fuentes = ['Ecopetrol Nacional', 'Ecopetrol Importado', 'Termoyopal',
                 'Otros Productores', 'Importaciones G5', 'Importaciones Tumaco',
                 'Importaciones Plexa']
colores = [EIA['blue1'], EIA['blue2'], EIA['orange'], EIA['gray'],
           EIA['green'], EIA['red'], EIA['purple']]

fig, ax = plt.subplots(figsize=(11, 6))

bottom = np.zeros(len(df_plot))
for fuente, color in zip(lista_fuentes, colores):
    vals = df_plot[fuente].values
    ax.bar(df_plot.index, vals, bottom=bottom, color=color, label=fuente, width=22)
    bottom += vals

ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax.xaxis.set_minor_locator(mdates.MonthLocator())
ax.set_ylabel('Toneladas / mes', fontsize=9)
totales = df_plot[lista_fuentes].sum(axis=1)
ax.set_ylim(0, totales.max() * 1.15)
ax.legend(ncol=4, loc='upper left', frameon=False)

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_color('#BDBDBD')
ax.spines['bottom'].set_color('#BDBDBD')
ax.yaxis.grid(True, linestyle='--', alpha=0.5, color='gray')

ax.set_title('Oferta nacional mensual de GLP por fuente (toneladas / mes)  |  @SergioCabrales',
             fontsize=13, color='#1A1A1A', pad=10)

fig.text(0.4, 0.01, 'Fuente: Ecopetrol y otras fuentes, cálculos GASNOVA',
         fontsize=7, color=EIA['gray'])

plt.tight_layout()
plt.show()

assert np.allclose(totales.values, oferta.loc[df_plot.index, 'Total'].values, atol=1e-6)
print('OK: coincide con el total de la lámina de oferta nacional vs. importado.')

In [ ]:
# Oferta nacional por fuente y por año, en promedio ton/mes (año parcial comparable).
apply_eia_theme()

df_anual = promedio_mensual_anual(fuentes)
lista_fuentes = ['Ecopetrol Nacional', 'Ecopetrol Importado', 'Termoyopal',
                 'Otros Productores', 'Importaciones G5', 'Importaciones Tumaco',
                 'Importaciones Plexa']
colores = [EIA['blue1'], EIA['blue2'], EIA['orange'], EIA['gray'],
           EIA['green'], EIA['red'], EIA['purple']]

fig, ax = plt.subplots(figsize=(12, 5.5))

x = np.arange(len(df_anual))
bottom = np.zeros(len(df_anual))
totales = df_anual[lista_fuentes].sum(axis=1).values
for fuente, color in zip(lista_fuentes, colores):
    vals = df_anual[fuente].values
    ax.bar(x, vals, bottom=bottom, color=color, label=fuente, width=0.7)
    for xi, v, b in zip(x, vals, bottom):
        if v > totales.max() * 0.04:
            ax.text(xi, b + v / 2, f'{v:,.0f}', ha='center', va='center',
                    fontsize=7, color='white', fontweight='bold')
    bottom += vals

for xi, t in zip(x, totales):
    ax.text(xi, t + totales.max() * 0.015, f'{t:,.0f}',
            ha='center', va='bottom', fontsize=7.5, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(etiquetas_anuales(df_anual.index))
ax.set_ylabel('Toneladas / mes (promedio del año)', fontsize=9)
ax.set_ylim(0, totales.max() * 1.16)
ax.legend(ncol=4, loc='upper left', frameon=False)

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_color('#BDBDBD')
ax.spines['bottom'].set_color('#BDBDBD')
ax.yaxis.grid(True, linestyle='--', alpha=0.5, color='gray')

ax.set_title('Oferta nacional de GLP por fuente (promedio toneladas / mes) @SergioCabrales',
             fontsize=11, color='#1A1A1A', loc='left', pad=10)

fig.text(0.01, 0.01, 'Fuente: Ecopetrol y otras fuentes, cálculos GASNOVA.' + NOTA_PARCIAL,
         fontsize=7, color=EIA['gray'])

plt.tight_layout()
plt.show()

In [ ]:
# Promedio anual (ton/mes) abierto por fuente. Mismo agregado que la lámina
# mensual por fuente y que la de nacional vs. importado.
df_anual = promedio_mensual_anual(oferta[NACIONAL_DESAGREGADO + ['Importado']])

fig, ax = plt.subplots(figsize=(13, 7))

x = np.arange(len(df_anual))
COMPONENTES = NACIONAL_DESAGREGADO + ['Importado']
COLORES = {
    'Campos':            '#1F77B4',
    'Barranca':          'darkred',
    'Cartagena':         'darkgreen',
    'Termoyopal':        '#8C564B',
    'Otros Productores': '#7F7F7F',
    'Importado':         '#FF7F0E',
}
ETIQUETAS = {'Barranca': 'Refinería Barranca', 'Cartagena': 'Refinería Cartagena'}

totales = df_anual[COMPONENTES].sum(axis=1).values
bottom = np.zeros(len(df_anual))
for comp in COMPONENTES:
    vals = df_anual[comp].values
    ax.bar(x, vals, bottom=bottom, width=0.8, alpha=0.6,
           color=COLORES[comp], label=ETIQUETAS.get(comp, comp))
    for xi, v, b in zip(x, vals, bottom):
        if v > totales.max() * 0.035:
            ax.text(xi, b + v / 2, f'{v:,.0f}', ha='center', va='center',
                    color='white', fontsize=8)
    bottom += vals

for xi, t in zip(x, totales):
    ax.text(xi, t + totales.max() * 0.015, f'{t:,.0f}',
            ha='center', va='bottom', fontsize=9, color='black')

ax.set_xticks(x)
ax.set_xticklabels(etiquetas_anuales(df_anual.index), fontsize=10)
ax.set_title('Oferta Anual Promedio de GLP en Colombia — Comparación por Fuente @SergioCabrales',
             fontsize=13, pad=14)
ax.set_ylabel('Toneladas / Mes (promedio del año)', fontsize=11)

ax.spines['right'].set_visible(False)
ax.spines['top'].set_visible(False)
ax.spines['left'].set_alpha(0.3)
ax.spines['bottom'].set_alpha(0.3)
ax.grid(axis='y', linestyle='--', linewidth=0.6, alpha=0.4, color='grey')
ax.set_axisbelow(True)

plt.yticks(fontsize=10)
ax.legend(frameon=False, loc='upper left', fontsize=10, ncol=2,
          title='Fuente de la oferta', title_fontsize=10)
ax.set_ylim(0, totales.max() * 1.25)

plt.tight_layout()
fig.text(0.01, 0.01,
         'Fuente: Gasnova, cálculos propios a partir de declaraciones de producción.' + NOTA_PARCIAL,
         fontsize=7, color='grey')
plt.show()

# Se guarda para la lámina siguiente, garantizando el mismo total
df_oferta_annual = promedio_mensual_anual(oferta[['Nacional', 'Importado', 'Total']])
df_oferta_annual['pct_importado'] = df_oferta_annual['Importado'] / df_oferta_annual['Total'] * 100
df_oferta_annual['pct_nacional']  = df_oferta_annual['Nacional']  / df_oferta_annual['Total'] * 100
assert np.allclose(totales, df_oferta_annual['Total'].values, atol=1e-6)
print('OK: el total anual apilado coincide con Nacional + Importado.')

In [ ]:
# Promedio anual (ton/mes): nacional vs. importado.
# `df_oferta_annual` viene de la lámina anterior, así que los totales coinciden.
fig, ax = plt.subplots(figsize=(12, 7))

x = np.arange(len(df_oferta_annual))
etiquetas = etiquetas_anuales(df_oferta_annual.index)

ax.bar(x, df_oferta_annual['Nacional'], label='Producción Nacional',
       color='#005F73', alpha=0.6, width=0.6)
ax.bar(x, df_oferta_annual['Importado'], bottom=df_oferta_annual['Nacional'],
       label='Importado', color='orange', alpha=0.6, width=0.6)

_max = df_oferta_annual['Total'].max()
for i in range(len(df_oferta_annual)):
    nac = df_oferta_annual['Nacional'].iloc[i]
    imp = df_oferta_annual['Importado'].iloc[i]
    tot = df_oferta_annual['Total'].iloc[i]
    if nac > 0:
        ax.text(i, nac / 2, f"{df_oferta_annual['pct_nacional'].iloc[i]:.1f}%",
                ha='center', va='center', color='white', fontsize=10)
    if imp > 0:
        ax.text(i, nac + imp / 2, f"{df_oferta_annual['pct_importado'].iloc[i]:.1f}%",
                ha='center', va='center', color='black', fontsize=10)
    ax.text(i, tot + _max * 0.03, f'{tot:,.0f}',
            ha='center', va='bottom', fontsize=9, color='black')

ax.set_xticks(x)
ax.set_xticklabels(etiquetas, rotation=45, ha='right', fontsize=10)
ax.set_title('Oferta Promedio Anual de GLP en Colombia @SergioCabrales', fontsize=13)
ax.set_ylabel('Toneladas / mes (promedio del año)', fontsize=11)

ax.spines['right'].set_visible(False)
ax.spines['top'].set_visible(False)
ax.grid(axis='y', linestyle='--', alpha=0.4, color='grey')
ax.set_axisbelow(True)

plt.yticks(fontsize=10)
ax.legend(frameon=False, loc='upper left', fontsize=10)
ax.set_ylim(0, _max * 1.20)

plt.tight_layout()
fig.text(0.01, 0.01,
         'Fuente: Gasnova, cálculos propios a partir de declaraciones de producción.' + NOTA_PARCIAL,
         fontsize=7, color='grey')
plt.show()

# Importación histórica de GL

In [ ]:
# Importaciones mensuales por fuente. La suma reproduce oferta['Importado'].
apply_eia_theme()
lista_fuentes = IMPORTADO
colores = [EIA['blue2'], EIA['green'], EIA['red'], EIA['purple']]
df_plot = importaciones[importaciones.index >= '2020-01-01']

fig, ax = plt.subplots(figsize=(11, 6))

bottom = np.zeros(len(df_plot))
for fuente, color in zip(lista_fuentes, colores):
    vals = df_plot[fuente].values
    ax.bar(df_plot.index, vals, bottom=bottom, color=color, label=fuente, width=22)
    bottom += vals

ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax.xaxis.set_minor_locator(mdates.MonthLocator())
ax.set_ylabel('Toneladas / mes', fontsize=9)
totales = df_plot.sum(axis=1)
ax.set_ylim(0, totales.max() * 1.15)
ax.legend(ncol=4, loc='upper left', frameon=False)

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_color('#BDBDBD')
ax.spines['bottom'].set_color('#BDBDBD')
ax.yaxis.grid(True, linestyle='--', alpha=0.5, color='gray')

ax.set_title('Importaciones mensuales de GLP por fuente (toneladas / mes)  @SergioCabrales',
             fontsize=13, color='#1A1A1A', pad=10)
fig.text(0.01, 0.01, 'Fuente: Ecopetrol y otras fuentes, cálculos GASNOVA',
         fontsize=7, color=EIA['gray'])

plt.tight_layout()
plt.show()

assert np.allclose(totales.values, oferta.loc[df_plot.index, 'Importado'].values, atol=1e-6)
print('OK: coincide con la barra "Importado" de las láminas de oferta.')

In [ ]:
# Importaciones por año, en promedio ton/mes (año parcial comparable).
apply_eia_theme()

df_anual = promedio_mensual_anual(importaciones)
totales  = df_anual.sum(axis=1)
particip = df_anual.div(totales.replace(0, np.nan), axis=0) * 100

lista_fuentes = IMPORTADO
colores = [EIA['blue2'], EIA['green'], EIA['red'], EIA['purple']]

fig, ax = plt.subplots(figsize=(12, 5))

x = np.arange(len(df_anual))
bottom = np.zeros(len(df_anual))
for fuente, color in zip(lista_fuentes, colores):
    vals = df_anual[fuente].values
    pcts = particip[fuente].values
    ax.bar(x, vals, bottom=bottom, color=color, label=fuente, width=0.7)
    for xi, v, b, p in zip(x, vals, bottom, pcts):
        if v > totales.max() * 0.045 and np.isfinite(p):
            ax.text(xi, b + v / 2, f'{p:,.0f}%', ha='center', va='center',
                    fontsize=7, color='white', fontweight='bold')
    bottom += vals

for xi, t in zip(x, totales.values):
    ax.text(xi, t + totales.max() * 0.015, f'{t:,.0f}',
            ha='center', va='bottom', fontsize=7.5, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(etiquetas_anuales(df_anual.index))
ax.set_ylabel('Toneladas / mes (promedio del año)', fontsize=9)
ax.set_ylim(0, totales.max() * 1.14)
ax.legend(ncol=4, loc='upper left', frameon=False)

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_color('#BDBDBD')
ax.spines['bottom'].set_color('#BDBDBD')
ax.yaxis.grid(True, linestyle='--', alpha=0.5, color='gray')

ax.set_title('Importaciones de GLP por fuente  @SergioCabrales',
             fontsize=13, color='#1A1A1A', pad=10)
fig.text(0.01, 0.01,
         'Fuente: Ecopetrol y otras fuentes, cálculos GASNOVA. '
         'Etiquetas: % del total importado del año.' + NOTA_PARCIAL,
         fontsize=7, color=EIA['gray'])

plt.tight_layout()
plt.show()

# Consumo de GLP por tipo de canal de distribución

In [ ]:
apply_eia_theme()

# Datos: consolidar bloques anuales de la pestaña en una serie mensual
raw = pd.read_excel(RUTA_GASNOVA, sheet_name="12.Demanda nacional-canal", header=None)

meses = {m: i for i, m in enumerate(MESES_ES, start=1)}

registros, anio = [], None
for _, fila in raw.iterrows():
    celda = str(fila[0]).strip().lower()
    if celda.startswith("datos"):
        anio = int(celda.split()[-1])
    elif celda in meses and anio is not None and pd.notna(fila[3]):
        registros.append({"Fecha": pd.Timestamp(anio, meses[celda], 1),
                          "Envasado": fila[3], "Granel": fila[4], "Redes *": fila[5]})

demanda_canal = (pd.DataFrame(registros).set_index("Fecha").sort_index()
                 .apply(pd.to_numeric, errors="coerce").fillna(0.0) / 1e3)  # ton/mes

df_plot = demanda_canal[demanda_canal.index >= "2020-01-01"]
canales = ["Envasado", "Granel", "Redes *"]
colores = [EIA["blue1"], EIA["orange"], EIA["green"]]

fig, ax = plt.subplots(figsize=(11, 6))

bottom = np.zeros(len(df_plot))
for canal, color in zip(canales, colores):
    vals = df_plot[canal].values
    ax.bar(df_plot.index, vals, bottom=bottom, color=color, label=canal, width=22)
    bottom += vals

ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax.xaxis.set_minor_locator(mdates.MonthLocator())
ax.set_ylabel("Toneladas / mes", fontsize=9)
totales = df_plot[canales].sum(axis=1)
ax.set_ylim(0, totales.max() * 1.12)
ax.legend(ncol=3, loc="upper left", frameon=False)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_color("#BDBDBD")
ax.spines["bottom"].set_color("#BDBDBD")
ax.yaxis.grid(True, linestyle='--', alpha=0.5, color='gray')

ax.set_title("Demanda nacional mensual de GLP por canal de venta @SergioCabrales",
             fontsize=13, color="#1A1A1A", pad=10)
fig.text(0.01, 0.01, "Fuente: SUI, cálculos GASNOVA", fontsize=7, color=EIA["gray"])

plt.tight_layout()
plt.show()

print(f'Demanda por canal: {demanda_canal.index.min():%Y-%m} a '
      f'{demanda_canal.index.max():%Y-%m}')

In [ ]:
# Demanda por canal y año, en promedio ton/mes (el año parcial deja de "caerse").
apply_eia_theme()

df = demanda_canal[demanda_canal.index >= "2012-01-01"]
df_anual = promedio_mensual_anual(df)
canales = ["Envasado", "Granel", "Redes *"]
colores = [EIA["blue1"], EIA["orange"], EIA["green"]]

fig, ax = plt.subplots(figsize=(11, 6))

x = np.arange(len(df_anual))
totales = df_anual[canales].sum(axis=1).values
bottom = np.zeros(len(df_anual))
for canal, color in zip(canales, colores):
    vals = df_anual[canal].values
    ax.bar(x, vals, bottom=bottom, color=color, label=canal, width=0.7)
    for xi, v, b in zip(x, vals, bottom):
        if v > totales.max() * 0.04:
            ax.text(xi, b + v / 2, f"{v:,.0f}", ha="center", va="center",
                    fontsize=7, color="white", fontweight="bold")
    bottom += vals

for xi, t in zip(x, totales):
    ax.text(xi, t + totales.max() * 0.015, f"{t:,.0f}",
            ha="center", va="bottom", fontsize=7.5, fontweight="bold")

ax.set_xticks(x)
ax.set_xticklabels(etiquetas_anuales(df_anual.index, referencia=df.index))
ax.set_ylabel("Toneladas / mes (promedio del año)", fontsize=9)
ax.set_ylim(0, totales.max() * 1.12)
ax.legend(ncol=3, loc="upper left", frameon=False)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_color("#BDBDBD")
ax.spines["bottom"].set_color("#BDBDBD")
ax.yaxis.grid(True, linestyle='--', alpha=0.5, color='gray')

ax.set_title("Demanda nacional de GLP por canal de venta @SergioCabrales",
             fontsize=13, color="#1A1A1A", pad=10)
fig.text(0.01, 0.01, "Fuente: SUI, cálculos GASNOVA." + nota_parcial(df.index),
         fontsize=7, color=EIA["gray"])

plt.tight_layout()
plt.show()

# Consumo de GLP por sector

In [ ]:
apply_eia_theme()

# Serie mensual por sector (fila de fechas 15; sectores filas 16-20)
raw = pd.read_excel(RUTA_GASNOVA, sheet_name="13. Comportamiento por sectores",
                    header=None)

fechas_sec = pd.to_datetime(raw.iloc[15, 3:], errors="coerce")
demanda_sector = raw.iloc[16:21, 3:].T
demanda_sector.columns = [str(c).strip() for c in raw.iloc[16:21, 2]]
demanda_sector.index = fechas_sec
demanda_sector = (demanda_sector.apply(pd.to_numeric, errors="coerce")
                  .dropna(how="all"))
demanda_sector = demanda_sector[demanda_sector.index.notna()].fillna(0.0)
demanda_sector = demanda_sector[demanda_sector.sum(axis=1) > 0] / 1e3   # ton/mes

df_prom = promedio_mensual_anual(demanda_sector)
sectores = ["Residencial", "Comercial", "Industrial", "Oficial", "Otros"]
colores  = [EIA["blue1"], EIA["orange"], EIA["gray"], EIA["red"], EIA["green"]]

fig, ax = plt.subplots(figsize=(11, 6))

x = np.arange(len(df_prom))
totales = df_prom[sectores].sum(axis=1).values
bottom = np.zeros(len(df_prom))
for sector, color in zip(sectores, colores):
    vals = df_prom[sector].values
    ax.bar(x, vals, bottom=bottom, color=color, label=sector, width=0.7)
    for xi, v, b in zip(x, vals, bottom):
        if v > totales.max() * 0.05:
            ax.text(xi, b + v / 2, f"{v:,.0f}", ha="center", va="center",
                    fontsize=7, color="white", fontweight="bold")
    bottom += vals

for xi, t in zip(x, totales):
    ax.text(xi, t + totales.max() * 0.015, f"{t:,.0f}",
            ha="center", va="bottom", fontsize=7.5, fontweight="bold")

ax.set_xticks(x)
ax.set_xticklabels(etiquetas_anuales(df_prom.index, referencia=demanda_sector.index))
ax.set_ylabel("Toneladas / mes (promedio del año)", fontsize=9)
ax.set_ylim(0, totales.max() * 1.14)
ax.legend(ncol=5, loc="upper left", frameon=False)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_color("#BDBDBD")
ax.spines["bottom"].set_color("#BDBDBD")
ax.yaxis.grid(True, linestyle='--', alpha=0.5, color='gray')

ax.set_title("Ventas promedio mes de GLP por sector @SergioCabrales",
             fontsize=13, color="#1A1A1A", pad=10)
fig.text(0.01, 0.01, "Fuente: SUI, cálculos GASNOVA." + nota_parcial(demanda_sector.index),
         fontsize=7, color=EIA["gray"])

plt.tight_layout()
plt.show()

# Delcaración de produccion GLP

https://www.minenergia.gov.co/es/misional/hidrocarburos/funcionamiento-del-sector/gas-licuado-de-petr%C3%B3leo/

In [ ]:
import requests

url = "https://www.minenergia.gov.co/documents/15888/Soporte_magn%C3%A9tico_declaraci%C3%B3n_de_producci%C3%B3n_de_GLP_2026-2030.xlsx"

headers = {
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/151.0.0.0 Safari/537.36"
    )
}

try:
    respuesta = requests.get(
        url,
        headers=headers,
        timeout=(30, 120)
    )

    respuesta.raise_for_status()

    with open("Anexo_Graficos.xlsx", "wb") as archivo:
        archivo.write(respuesta.content)

    print("Archivo descargado exitosamente.")

except requests.exceptions.Timeout:
    print("La conexión con MinEnergía agotó el tiempo de espera.")

except requests.exceptions.RequestException as e:
    print(f"Error al descargar el archivo: {e}")

In [ ]:
# Cargar el archivo Excel
excel_path = "Anexo_Graficos.xlsx"
xls = pd.ExcelFile(excel_path)
# Mostrar los nombres de las pestañas
sheet_names = xls.sheet_names
sheet_names

In [ ]:
# Crear un DataFrame vacío para consolidar la información
df_consolidado = pd.DataFrame()

# Iterar sobre las hojas
for sheet in sheet_names:
    df = xls.parse(sheet)

    # Buscar la columna con "mes" y con "PP (TON/MES)" o "CIDV (TON/MES)"
    col_mes = next((col for col in df.columns if re.search(r'mes', str(col), re.IGNORECASE)), None)
    col_valor = next((col for col in df.columns if re.search(r'(PP|CIDV) \(TON/MES\)', str(col), re.IGNORECASE)), None)

    # Continuar solo si ambas columnas existen
    if col_mes and col_valor:
        temp_df = df[[col_mes, col_valor]].copy()
        temp_df.columns = ['Mes', sheet]

        # Convertir 'Mes' a datetime
        try:
            temp_df['Mes'] = pd.to_datetime(temp_df['Mes'], errors='coerce')
        except:
            continue

        # Agregar los datos al DataFrame consolidado
        if df_consolidado.empty:
            df_consolidado = temp_df.set_index('Mes')
        else:
            df_consolidado = df_consolidado.merge(temp_df.set_index('Mes'), left_index=True, right_index=True, how='outer')

df_consolidado = df_consolidado[df_consolidado.index.notna()]

df_consolidado

In [ ]:
# Asegurarse de que la columna 'Mes' sea el índice del DataFrame
df_consolidado = df_consolidado.reset_index().set_index('Mes')

# Reemplazar valores NaN por cero
df_consolidado = df_consolidado.fillna(0)

df_consolidado

In [ ]:
# Declaración de producción MinEnergía: nacional vs. importado
IMPORTADORES = ['Montagas', 'Inversiones GLP', 'Colgas', 'Chilco']

oferta_importada_cols = [c for c in df_consolidado.columns if c in IMPORTADORES]
oferta_nacional_cols  = [c for c in df_consolidado.columns if c not in IMPORTADORES]

_faltan = [c for c in IMPORTADORES if c not in df_consolidado.columns]
if _faltan:
    print(f'AVISO: no se encontraron las pestañas de importación {_faltan}; '
          'quedarían clasificadas como oferta nacional.')

oferta_nacional  = df_consolidado[oferta_nacional_cols].sum(axis=1)
oferta_importada = df_consolidado[oferta_importada_cols].sum(axis=1)

fig, ax = plt.subplots(figsize=(12, 6))

ax.plot(oferta_nacional.index, oferta_nacional, marker='o', label='Oferta Nacional')
ax.plot(oferta_importada.index, oferta_importada, marker='s', label='Oferta Importada')
ax.set_title('Oferta Nacional vs. Oferta Importada de GLP (TON/MES)')
ax.set_xlabel('Fecha')
ax.set_ylabel('Toneladas por mes')

ax.spines['right'].set_visible(False)
ax.spines['top'].set_visible(False)
ax.grid(True, linestyle='--', alpha=0.4)
ax.legend(frameon=False)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Preparar los datos en el formato esperado
datos = pd.DataFrame({
    "Total cantidad ofrecida nacional": oferta_nacional,
    "Cantidad importada": oferta_importada
})
datos

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(datos))
bar_width = 0.8

color_nacional  = '#1f77b4'
color_importado = '#ff7f0e'

ax.bar(x, datos["Total cantidad ofrecida nacional"], label="GLP nacional",
       color=color_nacional, alpha=0.4, width=bar_width)
ax.bar(x, datos["Cantidad importada"],
       bottom=datos["Total cantidad ofrecida nacional"], label="GLP importado",
       color=color_importado, alpha=0.4, width=bar_width)

ax.set_ylabel("Oferta de GLP (ton/mes)", fontsize=10)
ax.set_title("Oferta mensual de GLP en Colombia @SergioCabrales", fontsize=12)
ax.tick_params(axis='y', labelsize=8)
ax.legend(frameon=False)

ax.set_xticks(x)
ax.set_xticklabels(datos.index.strftime('%Y-%m'), rotation=45, ha='right', fontsize=8)

ax.spines['right'].set_visible(False)
ax.spines['top'].set_visible(False)

plt.figtext(0.5, -0.04, "Fuente de los datos: Cálculos propios a partir de declaraciones de producción",
            ha="center", fontsize=8, color='grey')

plt.tight_layout()
plt.show()

print(f'Oferta declarada: {datos.index.min():%Y-%m} a {datos.index.max():%Y-%m} '
      f'({len(datos)} meses)')

In [ ]:
# Proyectos nuevos sobre la oferta declarada.
# Antes los proyectos terminaban en 2029-12 mientras `datos` llegaba a 2030-12,
# por lo que el total se desplomaba en el último año sin razón. Ahora el
# horizonte de los proyectos se extiende hasta el final de la serie.
fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(datos))
bar_width = 0.8

color_nacional  = '#1f77b4'
color_importado = '#ff7f0e'
color_okianus   = '#2ca02c'
color_okianus2  = '#1B9E77'
color_frontera  = '#d62728'

ax.bar(x, datos["Total cantidad ofrecida nacional"], label="GLP nacional",
       color=color_nacional, alpha=0.4, width=bar_width)
ax.bar(x, datos["Cantidad importada"], bottom=datos["Total cantidad ofrecida nacional"],
       label="GLP importado", color=color_importado, alpha=0.4, width=bar_width)

# Entrada de proyectos: (inicio, ton/mes)
PROYECTOS = {
    "Okianus - Fase 1": ("2026-01-01",  4000, color_okianus),
    "Okianus - Fase 2": ("2026-11-01",  9000, color_okianus2),
    "Frontera":         ("2026-01-01", 40000, color_frontera),
}
FIN_HORIZONTE = datos.index.max()

proyectos = pd.DataFrame(0.0, index=datos.index, columns=list(PROYECTOS))
for nombre, (inicio, cap, _c) in PROYECTOS.items():
    proyectos.loc[(datos.index >= inicio) & (datos.index <= FIN_HORIZONTE), nombre] = cap

bottom = (datos["Total cantidad ofrecida nacional"] + datos["Cantidad importada"]).values
for nombre, (_i, _cap, color) in PROYECTOS.items():
    vals = proyectos[nombre].values
    ax.bar(x, vals, bottom=bottom, label=nombre, color=color, alpha=0.6, width=bar_width)
    bottom = bottom + vals

ax.set_ylabel("Oferta de GLP (ton/mes)", fontsize=10)
ax.set_title("Oferta mensual de GLP @SergioCabrales", fontsize=12)
ax.tick_params(axis='y', labelsize=8)
ax.legend(frameon=False)

ax.set_xticks(x)
ax.set_xticklabels(datos.index.strftime('%Y-%m'), rotation=45, ha='right', fontsize=8)

ax.spines['right'].set_visible(False)
ax.spines['top'].set_visible(False)

plt.figtext(0.5, -0.04, "Fuente de los datos: Cálculos propios a partir de declaraciones de producción",
            ha="center", fontsize=8, color='grey')

plt.tight_layout()
plt.show()

In [ ]:
# CAPACIDAD de importación por puerto (no volumen efectivamente importado).
# Se mantiene en un DataFrame aparte para no mezclarla con `datos`, que
# contiene volúmenes declarados; el total de esta lámina NO es comparable con
# el de las láminas de oferta efectiva.
cap = {
    "G4 (Inversiones, Chilco, Rayogas, Montagas)": {
        "actual": 13000, "2026_01_10": 13000, "2026_11_12": 13000,
        "2027": 13000, "2028": 13000, "2029": 13000, "2030": 13000},
    "Colgas": {
        "actual": 17000, "2026_01_10": 25000, "2026_11_12": 32000,
        "2027": 32000, "2028": 32000, "2029": 32000, "2030": 32000},
    "Plexa Port": {
        "actual": 17000, "2026_01_10": 35000, "2026_11_12": 35000,
        "2027": 35000, "2028": 35000, "2029": 35000, "2030": 35000},
    "Puerto Bahía": {
        "actual": 0, "2026_01_10": 5000, "2026_11_12": 15000,
        "2027": 15000, "2028": 42000, "2029": 42000, "2030": 42000},
}

idx = datos.index

def serie_cap(p):
    s = pd.Series(float(p["actual"]), index=idx)
    s[(idx >= "2026-01-01") & (idx <= "2026-10-31")] = p["2026_01_10"]
    s[(idx >= "2026-11-01") & (idx <= "2026-12-31")] = p["2026_11_12"]
    for anio in ("2027", "2028", "2029", "2030"):
        s[(idx >= f"{anio}-01-01") & (idx <= f"{anio}-12-31")] = p[anio]
    return s

col_g4, col_col = "Imp - G4", "Imp - Colgas"
col_plex, col_pb = "Imp - Plexa Port", "Imp - Puerto Bahía"

capacidad = pd.DataFrame({
    col_g4:   serie_cap(cap["G4 (Inversiones, Chilco, Rayogas, Montagas)"]),
    col_col:  serie_cap(cap["Colgas"]),
    col_plex: serie_cap(cap["Plexa Port"]),
    col_pb:   serie_cap(cap["Puerto Bahía"]),
})

x = np.arange(len(datos))
bar_width = 0.8

color_nacional = '#1f77b4'
color_g4       = '#ff7f0e'
color_colgas   = '#ffbb78'
color_plexa    = 'green'
color_pbahia   = 'red'
COLS_CAP    = [col_g4, col_col, col_plex, col_pb]
COLORES_CAP = [color_g4, color_colgas, color_plexa, color_pbahia]
NOMBRES_CAP = ["Importado - G4 (Inversiones, Chilco, Rayogas, Montagas)",
               "Importado - Colgas", "Importado - Plexa Port",
               "Importado - Puerto Bahía"]

fig, ax = plt.subplots(figsize=(12, 6))

ax.bar(x, datos["Total cantidad ofrecida nacional"],
       label="GLP nacional (declarado)", color=color_nacional, alpha=0.6, width=bar_width)

bottom = datos["Total cantidad ofrecida nacional"].values
for c, color, nombre in zip(COLS_CAP, COLORES_CAP, NOMBRES_CAP):
    ax.bar(x, capacidad[c].values, bottom=bottom, label=nombre,
           color=color, alpha=0.6, width=bar_width)
    bottom = bottom + capacidad[c].values

ax.set_ylabel("Oferta de GLP (ton/mes)", fontsize=10)
ax.set_title("Producción nacional declarada + capacidad de importación por puerto @SergioCabrales",
             fontsize=12)
ax.tick_params(axis='y', labelsize=8)
ax.legend(frameon=False, ncol=1)

ax.set_xticks(x)
ax.set_xticklabels(datos.index.strftime('%Y-%m'), rotation=45, ha='right', fontsize=8)

ax.spines['right'].set_visible(False)
ax.spines['top'].set_visible(False)

plt.figtext(0.5, -0.04,
            "Fuente: cálculos propios a partir de declaraciones de producción. "
            "Las barras naranjas/verde/roja son CAPACIDAD instalada de importación, "
            "no volumen importado: el total no es comparable con las láminas de oferta efectiva.",
            ha="center", fontsize=8, color='grey')

plt.tight_layout()
plt.show()

In [ ]:
# Oferta anual promedio (Gasnova), reutilizando `df_oferta_annual` ya calculado
fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(df_oferta_annual))

ax.bar(x, df_oferta_annual["Nacional"], label="GLP nacional", alpha=0.4)
ax.bar(x, df_oferta_annual["Importado"], bottom=df_oferta_annual["Nacional"],
       label="GLP importado", alpha=1)

for i in range(len(df_oferta_annual)):
    total = df_oferta_annual["Total"].iloc[i]
    pct   = df_oferta_annual["pct_importado"].iloc[i]
    ax.text(i, total * 1.01, f"{pct:.1f}%", ha="center", va="bottom", fontsize=11)

ax.spines["right"].set_visible(False)
ax.spines["top"].set_visible(False)
ax.set_ylabel("Oferta promedio (ton/mes)")
ax.set_title("Oferta anual promedio de GLP @SergioCabrales")
ax.set_xticks(x)
ax.set_xticklabels(etiquetas_anuales(df_oferta_annual.index))
ax.set_ylim(0, df_oferta_annual["Total"].max() * 1.15)
ax.legend(frameon=False)

plt.figtext(0.5, -0.04,
            "Fuente: Gasnova, cálculos propios a partir de declaraciones de producción."
            + NOTA_PARCIAL,
            ha="center", fontsize=8, color="gray")

plt.tight_layout()
plt.show()

In [ ]:
# Participación del importado (en %, no en razón: antes convivían
# `pct_importado` en % y `pct_importada` como cociente, y se leían como lo mismo)
df_oferta_annual[["Nacional", "Importado", "Total",
                  "pct_nacional", "pct_importado"]].round(1)

# Proyección de demanda de GLP

In [ ]:
# Escenarios de demanda UPME (ton/mes), indexados por fecha.
# El índice por fecha es lo que permite alinearlos con la oferta declarada:
# ambas series tienen 60 meses pero NO son los mismos 60 meses.
fechas_upme = pd.date_range("2025-01-01", periods=60, freq="MS")

esc_1 = [
    60.78, 58.93, 62.29, 61.68, 63.65, 63.58, 63.63, 61.09, 60.24, 63.57, 61.46, 61.59,
    61.05, 59.20, 62.56, 61.95, 63.93, 63.86, 63.91, 61.35, 60.51, 63.84, 61.73, 61.85,
    61.32, 59.47, 62.83, 62.23, 64.21, 64.14, 64.19, 61.63, 60.78, 64.11, 62.00, 62.13,
    61.57, 59.72, 63.08, 62.49, 64.46, 64.39, 64.44, 61.87, 61.02, 64.36, 62.24, 62.37,
    61.79, 59.94, 63.30, 62.71, 64.68, 64.62, 64.66, 62.09, 61.24, 64.58, 62.46, 62.59
]

escenarios = pd.DataFrame({"Escenario 1": esc_1}, index=fechas_upme) * 1000
escenarios.index.name = "Fecha"
escenarios.head()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

ax.plot(escenarios.index, escenarios["Escenario 1"], color='green',
        linewidth=2, label="Proyección")

ax.set_title("Proyección de demanda de GLP para el Escenario 1 - Tendencial")
ax.set_ylabel("ton/mes")
ax.set_ylim(0, 75000)
ax.grid(axis='y', linestyle='--', alpha=0.5)
ax.legend(frameon=False)
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

ax.spines['right'].set_visible(False)
ax.spines['top'].set_visible(False)

plt.tight_layout()
plt.figtext(0.5, -0.05, "Fuente: Datos UPME", ha="center", fontsize=9)
plt.show()

In [ ]:
escenarios["Escenario 2"] = np.array([
    63.012, 61.170, 64.527, 63.916, 65.891, 65.817, 65.867, 63.324, 62.479, 65.806, 63.693, 63.824,
    63.442, 61.595, 64.955, 64.350, 66.325, 66.253, 66.302, 63.750, 62.903, 66.235, 64.120, 64.250,
    63.921, 62.070, 65.433, 64.833, 66.808, 66.738, 66.786, 64.225, 63.376, 66.713, 64.596, 64.725,
    64.441, 62.587, 65.951, 65.357, 67.331, 67.263, 67.310, 64.741, 63.891, 67.231, 65.113, 65.241,
    65.020, 63.164, 66.527, 65.940, 67.911, 67.846, 67.891, 65.315, 64.464, 67.806, 65.688, 65.816
]) * 1000

In [ ]:
escenarios["Escenario 3"] = np.array([
    68.80, 66.96, 70.32, 69.71, 71.68, 71.61, 71.66, 69.11, 68.27, 71.59, 69.48, 69.61,
    69.41, 67.57, 70.93, 70.32, 72.30, 72.22, 72.27, 69.72, 68.87, 72.21, 70.09, 70.22,
    70.08, 68.23, 71.60, 71.00, 72.97, 72.90, 72.95, 70.39, 69.54, 72.88, 70.76, 70.89,
    70.80, 68.95, 72.31, 71.72, 73.69, 73.63, 73.67, 71.10, 70.25, 73.59, 71.48, 71.60,
    71.59, 69.74, 73.10, 72.51, 74.48, 74.42, 74.46, 71.89, 71.04, 74.38, 72.26, 72.39
]) * 1000

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

ESTILO_ESC = {
    "Escenario 1": dict(color='#1f77b4', linestyle='dotted'),
    "Escenario 2": dict(color='#2ca02c', linestyle='solid'),
    "Escenario 3": dict(color='#d62728', linestyle='dotted'),
}
for col, est in ESTILO_ESC.items():
    ax.plot(escenarios.index, escenarios[col], linewidth=2,
            label=f"Demanda - {col}", **est)

ax.set_title("Proyección de demanda de GLP @SergioCabrales")
ax.set_ylabel("ton/mes")
ax.set_ylim(0, 78000)
ax.grid(axis='y', linestyle='--', alpha=0.5)
ax.legend(frameon=False)
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

ax.spines['right'].set_visible(False)
ax.spines['top'].set_visible(False)

plt.tight_layout()
plt.figtext(0.5, -0.05, "Fuente: Elaboración UPME", ha="center", fontsize=9)
plt.show()

## Eje común oferta ↔ demanda

In [ ]:
# ── Eje común oferta ↔ demanda ────────────────────────────────────────────────
# `datos` (declaración de producción) cubre 2026-01→2030-12 y `escenarios`
# (UPME) cubre 2025-01→2029-12. Ambas series tienen 60 meses, así que al
# graficarlas por posición (x = range(len(datos))) la demanda quedaba desplazada
# 12 meses: la demanda de enero-2025 se dibujaba sobre la barra de enero-2026.
# Aquí se construye un eje de fechas común y se reindexan las dos series.

EJE       = datos.index.union(escenarios.index)
POS       = np.arange(len(EJE))
ETIQ_EJE  = EJE.strftime('%Y-%m')

oferta_eje = datos.reindex(EJE)
esc_eje    = escenarios.reindex(EJE)

def eje_fechas(ax, cada=3):
    ax.set_xticks(POS[::cada])
    ax.set_xticklabels(ETIQ_EJE[::cada], rotation=45, ha='right', fontsize=8)
    ax.set_xlim(-0.8, len(EJE) - 0.2)

print(f'Oferta declarada : {datos.index.min():%Y-%m} → {datos.index.max():%Y-%m} '
      f'({len(datos)} meses)')
print(f'Demanda UPME     : {escenarios.index.min():%Y-%m} → {escenarios.index.max():%Y-%m} '
      f'({len(escenarios)} meses)')
print(f'Eje común        : {EJE.min():%Y-%m} → {EJE.max():%Y-%m} ({len(EJE)} meses)')
_solape = datos.index.intersection(escenarios.index)
print(f'Meses con oferta y demanda simultáneas: {len(_solape)} '
      f'({_solape.min():%Y-%m} → {_solape.max():%Y-%m})')

In [ ]:
# Oferta declarada vs. demanda proyectada, alineadas por fecha
fig, ax = plt.subplots(figsize=(13, 6))

bar_width = 0.8
color_nacional  = '#1f77b4'
color_importado = '#ff7f0e'

nac = oferta_eje["Total cantidad ofrecida nacional"].values
imp = oferta_eje["Cantidad importada"].values

ax.bar(POS, nac, label="GLP nacional", color=color_nacional, alpha=0.4, width=bar_width)
ax.bar(POS, imp, bottom=nac, label="GLP importado", color=color_importado,
       alpha=0.4, width=bar_width)

ax.plot(POS, esc_eje["Escenario 2"].values, color='#2ca02c', linewidth=2,
        label="Demanda - Escenario 2")

# Déficit = demanda − oferta, sobre los meses en que existen ambas
_oferta_total = oferta_eje[["Total cantidad ofrecida nacional",
                            "Cantidad importada"]].sum(axis=1, min_count=1)
deficit = (esc_eje["Escenario 2"] - _oferta_total).dropna()
if (deficit > 0).any():
    _d = deficit[deficit > 0]
    print(f'Déficit frente al Escenario 2 en {len(_d)} meses; '
          f'máximo {_d.max():,.0f} ton/mes en {_d.idxmax():%Y-%m}')
else:
    print('Sin déficit frente al Escenario 2 en los meses con ambas series.')

ax.set_ylabel("Oferta de GLP (ton/mes)", fontsize=10)
ax.set_title("Oferta mensual de GLP en Colombia vs. demanda proyectada @SergioCabrales",
             fontsize=12)
ax.tick_params(axis='y', labelsize=8)
ax.legend(frameon=False)
eje_fechas(ax)

ax.spines['right'].set_visible(False)
ax.spines['top'].set_visible(False)

plt.figtext(0.5, -0.04, "Fuente de los datos: Cálculos propios a partir de declaraciones de producción",
            ha="center", fontsize=8, color='grey')

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(13, 6))

bar_width = 0.8
nac = oferta_eje["Total cantidad ofrecida nacional"].values
imp = oferta_eje["Cantidad importada"].values

ax.bar(POS, nac, label="GLP nacional", color='#1f77b4', alpha=0.4, width=bar_width)
ax.bar(POS, imp, bottom=nac, label="GLP importado", color='#ff7f0e',
       alpha=0.4, width=bar_width)

for col, est in ESTILO_ESC.items():
    ax.plot(POS, esc_eje[col].values, linewidth=2, label=f"Demanda - {col}", **est)

ax.set_ylabel("Oferta de GLP (ton/mes)", fontsize=10)
ax.set_title("Oferta mensual de GLP en Colombia vs. escenarios de demanda", fontsize=12)
ax.tick_params(axis='y', labelsize=8)
ax.legend(frameon=True, fancybox=False, shadow=False, facecolor="white")
eje_fechas(ax)

ax.spines['right'].set_visible(False)
ax.spines['top'].set_visible(False)

plt.figtext(0.5, -0.04, "Fuente de los datos: Cálculos propios a partir de declaraciones de producción",
            ha="center", fontsize=8, color='grey')

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(13, 6))

bar_width = 0.8
color_okianus  = 'red'
color_okianus2 = 'darkred'
color_frontera = 'green'

nac = oferta_eje["Total cantidad ofrecida nacional"].values
imp = oferta_eje["Cantidad importada"].values

ax.bar(POS, nac, label="GLP nacional", color='#1f77b4', alpha=0.4, width=bar_width)
ax.bar(POS, imp, bottom=nac, label="GLP importado", color='#ff7f0e',
       alpha=0.4, width=bar_width)

# Proyectos definidos sobre el eje común y vigentes hasta el fin del horizonte
PROYECTOS_EJE = {
    "Okianus - Fase 1": ("2026-01-01",  4000, color_okianus),
    "Okianus - Fase 2": ("2026-11-01",  9000, color_okianus2),
    "Frontera":         ("2026-08-01", 40000, color_frontera),
}
proyectos_eje = pd.DataFrame(0.0, index=EJE, columns=list(PROYECTOS_EJE))
for nombre, (inicio, capm, _c) in PROYECTOS_EJE.items():
    proyectos_eje.loc[EJE >= inicio, nombre] = capm
# Sin oferta declarada no hay barra sobre la que apilar
proyectos_eje = proyectos_eje.where(oferta_eje["Total cantidad ofrecida nacional"].notna(), 0.0)

bottom = np.nan_to_num(nac) + np.nan_to_num(imp)
for nombre, (_i, _c, color) in PROYECTOS_EJE.items():
    vals = proyectos_eje[nombre].values
    ax.bar(POS, vals, bottom=bottom, label=nombre, color=color, alpha=0.6, width=bar_width)
    bottom = bottom + vals

for col, est in ESTILO_ESC.items():
    ax.plot(POS, esc_eje[col].values, linewidth=2, label=f"Demanda - {col}", **est)

ax.set_ylabel("Oferta de GLP (ton/mes)", fontsize=10)
ax.set_title("Oferta mensual de GLP @SergioCabrales", fontsize=12)
ax.tick_params(axis='y', labelsize=8)
ax.legend(frameon=False, ncol=2)
eje_fechas(ax)

ax.spines['right'].set_visible(False)
ax.spines['top'].set_visible(False)

plt.figtext(0.5, -0.04, "Fuente de los datos: Cálculos propios a partir de declaraciones de producción",
            ha="center", fontsize=8, color='grey')

plt.tight_layout()
plt.show()

In [ ]:
# Producción nacional declarada + CAPACIDAD de importación, vs. demanda
capacidad_eje = capacidad.reindex(EJE)

fig, ax = plt.subplots(figsize=(13, 6))
bar_width = 0.8

nac = oferta_eje["Total cantidad ofrecida nacional"].values
ax.bar(POS, nac, label="GLP nacional (declarado)", color=color_nacional,
       alpha=0.4, width=bar_width)

bottom = np.nan_to_num(nac)
for c, color, nombre in zip(COLS_CAP, COLORES_CAP, NOMBRES_CAP):
    vals = np.nan_to_num(capacidad_eje[c].values)
    ax.bar(POS, vals, bottom=bottom, label=nombre, color=color, alpha=0.4, width=bar_width)
    bottom = bottom + vals

for col, est in ESTILO_ESC.items():
    ax.plot(POS, esc_eje[col].values, linewidth=2, label=f"Demanda - {col}", **est)

ax.set_ylabel("Oferta de GLP (ton/mes)", fontsize=10)
ax.set_title("Producción nacional declarada + capacidad de importación @SergioCabrales",
             fontsize=12)
ax.tick_params(axis='y', labelsize=8)
ax.legend(frameon=False, ncol=2)
eje_fechas(ax)

ax.spines['right'].set_visible(False)
ax.spines['top'].set_visible(False)

plt.figtext(0.5, -0.04,
            "Fuente: cálculos propios a partir de declaraciones de producción. "
            "Las barras de importación son CAPACIDAD instalada, no volumen importado.",
            ha="center", fontsize=8, color='grey')

plt.tight_layout()
plt.show()

In [ ]:
# Oferta + proyectos vs. demanda GLP e industrial de gas natural
GBTUD_A_TON_MES = 162000   # equivalente en ton/mes de 250 GBTUD
TON_MES_IND_REG = 20000    # industriales regulados

y1 = esc_eje["Escenario 2"]
y2 = y1 + GBTUD_A_TON_MES

fig, ax = plt.subplots(figsize=(13, 6))
bar_width = 0.8

nac = oferta_eje["Total cantidad ofrecida nacional"].values
imp = oferta_eje["Cantidad importada"].values

ax.bar(POS, nac, label="GLP nacional", color=color_nacional, alpha=0.4, width=bar_width)
ax.bar(POS, imp, bottom=nac, label="GLP importado", color=color_importado,
       alpha=0.4, width=bar_width)

bottom = np.nan_to_num(nac) + np.nan_to_num(imp)
for nombre, (_i, _c, color) in PROYECTOS_EJE.items():
    vals = proyectos_eje[nombre].values
    ax.bar(POS, vals, bottom=bottom, label=nombre, color=color, alpha=0.4, width=bar_width)
    bottom = bottom + vals

ax.plot(POS, y1.values, color='#2ca02c', linewidth=2, label="Demanda - Escenario 2")
ax.plot(POS, y2.values, color='red', linewidth=2,
        label="Demanda + Industriales no regulado (250 GBTUD)")
ax.fill_between(POS, y1.values, y2.values, color='red', alpha=0.04)

ax.set_ylabel("Oferta de GLP (ton/mes)", fontsize=10)
ax.set_title("Oferta mensual de GLP versus demanda proyectada de GLP e industrial GN @SergioCabrales",
             fontsize=12)
ax.tick_params(axis='y', labelsize=8)
ax.legend(frameon=False, ncol=2)
eje_fechas(ax)

ax.spines['right'].set_visible(False)
ax.spines['top'].set_visible(False)

plt.figtext(0.5, -0.04, "Fuente de los datos: Cálculos propios a partir de declaraciones de producción",
            ha="center", fontsize=8, color='grey')

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(13, 6))
bar_width = 0.8

nac = oferta_eje["Total cantidad ofrecida nacional"].values
ax.bar(POS, nac, label="GLP nacional (declarado)", color=color_nacional,
       alpha=0.4, width=bar_width)

bottom = np.nan_to_num(nac)
for c, color, nombre in zip(COLS_CAP, COLORES_CAP, NOMBRES_CAP):
    vals = np.nan_to_num(capacidad_eje[c].values)
    ax.bar(POS, vals, bottom=bottom, label=nombre, color=color, alpha=0.4, width=bar_width)
    bottom = bottom + vals

ax.plot(POS, y1.values, color='#2ca02c', linewidth=2, label="Demanda - Escenario 2")
ax.plot(POS, y2.values, color='red', linewidth=2,
        label="Demanda + Industriales no regulado (250 GBTUD)")
ax.fill_between(POS, y1.values, y2.values, color='red', alpha=0.04)

ax.set_ylabel("Oferta de GLP (ton/mes)", fontsize=10)
ax.set_title("Producción nacional declarada + capacidad de importación @SergioCabrales",
             fontsize=12)
ax.tick_params(axis='y', labelsize=8)
ax.legend(frameon=False, ncol=2, loc='upper left')
eje_fechas(ax)
ax.set_ylim(0, 300000)

ax.spines['right'].set_visible(False)
ax.spines['top'].set_visible(False)

plt.figtext(0.5, -0.04,
            "Fuente: cálculos propios a partir de declaraciones de producción. "
            "Las barras de importación son CAPACIDAD instalada, no volumen importado.",
            ha="center", fontsize=8, color='grey')

plt.tight_layout()
plt.show()

In [ ]:
# Se mantiene POS como eje: antes se reasignaba x = df["Fecha_str"], de modo que
# los ticks quedaban en las fechas de la proyección y las etiquetas en las de la
# oferta (dos calendarios distintos en el mismo eje).
y3 = y2 + TON_MES_IND_REG

fig, ax = plt.subplots(figsize=(13, 6))
bar_width = 0.8

nac = oferta_eje["Total cantidad ofrecida nacional"].values
imp = oferta_eje["Cantidad importada"].values

ax.bar(POS, nac, label="GLP nacional", color=color_nacional, alpha=0.4, width=bar_width)
ax.bar(POS, imp, bottom=nac, label="GLP importado", color=color_importado,
       alpha=0.4, width=bar_width)

bottom = np.nan_to_num(nac) + np.nan_to_num(imp)
for nombre, (_i, _c, color) in PROYECTOS_EJE.items():
    vals = proyectos_eje[nombre].values
    ax.bar(POS, vals, bottom=bottom, label=nombre, color=color, alpha=0.4, width=bar_width)
    bottom = bottom + vals

ax.plot(POS, y1.values, color='#2ca02c', linewidth=2, label="Demanda - Escenario 2")
ax.plot(POS, y2.values, color='red', linewidth=2,
        label="Demanda + Industriales no regulado (250 GBTUD)")
ax.plot(POS, y3.values, color='darkred', linewidth=2,
        label="Demanda + Industriales no regulado + Industriales regulado")

ax.fill_between(POS, y1.values, y2.values, color='red', alpha=0.04)
ax.fill_between(POS, y2.values, y3.values, color='darkred', alpha=0.04)

ax.set_ylabel("Oferta de GLP (ton/mes)", fontsize=10)
ax.set_title("Oferta mensual de GLP versus demanda proyectada de GLP e industrial GN @SergioCabrales",
             fontsize=12)
ax.tick_params(axis='y', labelsize=8)
ax.legend(frameon=False, ncol=2)
eje_fechas(ax)

ax.spines['right'].set_visible(False)
ax.spines['top'].set_visible(False)

plt.figtext(0.5, -0.04, "Fuente de los datos: Cálculos propios a partir de declaraciones de producción",
            ha="center", fontsize=8, color='grey')

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(13, 6))
bar_width = 0.8

nac = oferta_eje["Total cantidad ofrecida nacional"].values
ax.bar(POS, nac, label="GLP nacional (declarado)", color=color_nacional,
       alpha=0.4, width=bar_width)

bottom = np.nan_to_num(nac)
for c, color, nombre in zip(COLS_CAP, COLORES_CAP, NOMBRES_CAP):
    vals = np.nan_to_num(capacidad_eje[c].values)
    ax.bar(POS, vals, bottom=bottom, label=nombre, color=color, alpha=0.4, width=bar_width)
    bottom = bottom + vals

ax.plot(POS, y1.values, color='#2ca02c', linewidth=2, label="Demanda - Escenario 2")
ax.plot(POS, y2.values, color='red', linewidth=2,
        label="Demanda + Industriales no regulado (250 GBTUD)")
ax.plot(POS, y3.values, color='darkred', linewidth=2,
        label="Demanda + Industriales no regulado + Industriales regulado")

ax.fill_between(POS, y1.values, y2.values, color='red', alpha=0.04)
ax.fill_between(POS, y2.values, y3.values, color='purple', alpha=0.08)

ax.set_ylabel("Oferta de GLP (ton/mes)", fontsize=10)
ax.set_title("Producción nacional declarada + capacidad de importación @SergioCabrales",
             fontsize=12)
ax.tick_params(axis='y', labelsize=8)
ax.legend(frameon=False, ncol=2, loc='upper left')
eje_fechas(ax)
ax.set_ylim(0, 320000)

ax.spines['right'].set_visible(False)
ax.spines['top'].set_visible(False)

plt.figtext(0.5, -0.04,
            "Fuente: cálculos propios a partir de declaraciones de producción. "
            "Las barras de importación son CAPACIDAD instalada, no volumen importado.",
            ha="center", fontsize=8, color='grey')

plt.tight_layout()
plt.show()